# Valutazione della qualità della Knowledge Base

Questo script analizza un insieme di chunk testuali e ne verifica la qualità come conoscenza strutturata.

In particolare:
- legge un database di frasi organizzate per topic;
- usa un modello LLM per estrarre fatti in forma strutturata (auto, colore, capitale, orario);
- normalizza e salva i fatti in formato ASP/Clingo;
- applica regole di ridondanza, conflitto e copertura;
- risolve il programma con `clingo` per individuare occorrenze duplicate, contraddizioni e riferimenti non soddisfatti;
- calcola metriche di qualità come ridondanza, conflitti e copertura.

L’obiettivo è verificare se la base di conoscenza costruita sia coerente, compatta e sufficientemente completa rispetto ai target definiti.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Iterable, Literal, Optional
from pydantic import BaseModel
from ollama import chat
import re

import clingo

- Database:
    Ogni chiave è un identificatore di chunk, e ogni valore è un testo naturale. I testi riguardano 3 temi: (auto, città/capitale, orari di servizio). Il chunk_12 vuoto è intenzionale: serve a verificare il caso in cui un riferimento ("es. orario del negozio di domenica") non è coperto da nessuna frase. Questa è una situazione utile per testare la parte di "copertura".
- Rules:
    La parte RULES contiene un programma ASP/Clingo. In ASP, si definiscono fatti (es. auto("mario", "citroen", "chunk_1")) e regole (deduzioni logiche a partire dai fatti). Il programma cerca di rispondere a 3 domande:
    - ridondanza: quante informazioni sono state ripetute tra più chunk?
    - conflitto: ci sono informazioni incompatibili sullo stesso oggetto?
    - copertura: mancano informazioni attese rispetto a dei riferimenti di controllo?
    
- Spiegazione regole
    - Ridondanza: unicita(auto, P, M) è vero se esiste almeno un fatto auto per persona P e modello M.occorrenze(auto, P, M, N) conta quanti chunk contengono quell'associazione. Se N > 1, allora la stessa informazione appare più volte.
    - Discordanza / conflitto: se lo stesso oggetto X ha due colori diversi Y e Z in due chunk diversi S1 e S2, allora c'è un conflitto. Equivalentemente: stessa persona con auto diversa? stessa capitale associata a paese diverso? stesso luogo con orari differenti?
    - Copertura: si dichiarano i target minimali attesi: per il tema capitale, ci si aspetta una conoscenza per Francia e Italia, per gli orari, ci si aspetta un riferimento per il negozio di domenica - se esiste almeno un fatto capitale per il paese X, esso è "coperto", altrimenti, se X era un riferimento atteso e non è stato trovato, viene marcato come non_coperto.

In [ ]:
DATABASE = {
    # Topic Auto
    "chunk_1": "Mario possiede una Citroen nera.",
    "chunk_2": "Dopo anni di onorato servizio, Mario è riuscito a comprare una Citroen nera.",
    "chunk_3": "Una bellissima Citroen di colore nero è stata acquistata dal signor Mario due settimane fa.",
    "chunk_4": "Mario, lavoratore scrupoloso, decise di comprare una Citroen verde.",
    # Topic Città
    "chunk_5": "Una delle città più popolose dell'Inghilterra è Manchester.",
    "chunk_6": "La capitale della Francia è Parigi.",
    "chunk_7": "La Francia è uno stato con capitale Bordeaux.",
    # Topic Orari di servizio
    "chunk_8": "Il negozio è aperto dal lunedì al venerdì, dalle 9 alle 18.",
    "chunk_9": "Dal lunedì al venerdì l'orario di apertura del negozio va dalle 9:00 alle 18:00.",
    "chunk_10": "Il sabato il negozio è aperto dalle 10 alle 13.",
    "chunk_11": "Il negozio, il sabato, resta aperto dalle 10:00 fino alle 16:00.",
    "chunk_12": "",  # buco voluto: nessuna frase parla della domenica -> se hai una query di riferimento "orario domenica", risulterà scoperta
}

MODEL = "qwen2.5:7b-instruct"

RULES = """
    % Ridondanza
    unicita(auto, P, M) :- auto(P, M, _).
    occorrenze(auto, P, M, N) :- unicita(auto, P, M), N = #count{S : auto(P, M, S)}.

    unicita(colore, X, Y) :- colore(X, Y, _).
    occorrenze(colore, X, Y, N) :- unicita(colore, X, Y), N = #count{S : colore(X, Y, S)}.

    unicita(orario, Na, G, HI, HF) :- orario(Na, G, HI, HF, _).
    occorrenze(orario, Na, G, HI, HF, N) :- unicita(orario, Na, G, HI, HF), N = #count{S : orario(Na, G, HI, HF, S)}.

    % Discordanza
    discorde(colore, X, Y, Z, S1, S2) :- colore(X, Y, S1), colore(X, Z, S2), Y != Z, S1 < S2.
    discorde(capitale, X, Y, Z, S1, S2) :- capitale(X, Y, S1), capitale(X, Z, S2), Y != Z, S1 < S2.
    discorde(orario, N, G, H, F1, F2, S1, S2) :- orario(N, G, H, F1, S1), orario(N, G, H, F2, S2), F1 != F2, S1 < S2.

    % Copertura
    riferimento(capitale, "francia").
    riferimento(capitale, "italia").
    riferimento(orario, "negozio", "domenica").

    coperto(capitale, X) :- capitale(X, _, _).
    non_coperto(capitale, X) :- riferimento(capitale, X), not coperto(capitale, X).

    coperto(orario, N, G) :- orario(N, G, _, _, _).
    non_coperto(orario, N, G) :- riferimento(orario, N, G), not coperto(orario, N, G).

    % Output
    #show occorrenze/4.
    #show occorrenze/6.
    #show discorde/6.
    #show discorde/8.
    #show non_coperto/2.
    #show non_coperto/3.
    #show riferimento/2.
    #show riferimento/3.
"""

def norm_string(string: str) -> str:
    s = string.strip().lower()
    for t in TITOLI:
        if s.startswith(t):
            s = s[len(t):]
            break
    return s.strip()

def norm_hour(string: str) -> str:
    s = string.strip()
    if re.fullmatch(r"\d{1,2}", s):
        return f"{int(s):02d}:00"

    if re.fullmatch(r"\d{1,2}:\d{2}", s):
        h, m = s.split(":")
        return f"{int(h):02d}:{m}"
    return s


COLORI_CANONICI = {
    "nero": "nera",
    "bianco": "bianca",
    "rosso": "rossa",
    "verde": "verde"
}

def norm_color(colore: str) -> str:
    c = norm_string(colore)
    return COLORI_CANONICI.get(c, c)

In [ ]:
class Fact(BaseModel):
    tipo: Literal["auto", "colore", "capitale", "orario"]
    arg1: str
    arg2: str
    arg3: Optional[str] = None
    arg4: Optional[str] = None

class ExtractionResult(BaseModel):
    facts: list[Fact] = []


@dataclass(frozen=True)
class ClingoResults:
    occorrenze: list[clingo.Symbol] = field(default_factory=list)
    discordanze: list[clingo.Symbol] = field(default_factory=list)
    non_coperti: list[clingo.Symbol] = field(default_factory=list)
    riferimenti: list[clingo.Symbol] = field(default_factory=list)

@dataclass(frozen=True)
class QualityMetrics:
    total_facts: int
    total_references: int
    redundant_facts: int
    conflicts: int
    uncovered: int
    redundancy_rate: float
    conflict_rate: float
    coverage_rate: float

## Spiegazione delle funzioni

- escape_asp_str(value): Prepara una stringa per inserirla tra virgolette in un fatto ASP: sostituisce le virgolette doppie con la sequenza di escape `\"` e rimuove gli spazi iniziali e finali.
- extract_facts_by_chunk(chunk_id, text, model=MODEL): Estrae i fatti da un singolo chunk usando il modello LLM. Se il testo è vuoto, restituisce una lista vuota. Altrimenti invia al modello le istruzioni sui tipi di fatto ammessi, valida la risposta con `ExtractionResult` e converte ogni fatto in un atomo ASP. Se l’estrazione o la validazione fallisce, stampa un messaggio di errore e restituisce una lista vuota.
- build_extracted_llm(database): Applica `extract_facts_by_chunk` a ogni coppia identificatore-testo del database. Restituisce un dizionario in cui ogni chunk è associato alla lista dei fatti estratti.

In [ ]:
def escape_asp_str(value: str) -> str:
    return value.replace('"', '\\"').strip()

def extract_facts_by_chunk(chunk_id: str, text: str, model: str = MODEL) -> list[str]:
    if not text.strip():
        return []

    prompt = f"""
        Sei un esperto che si occupa di estrarre fatti da frasi.
        Estrai SOLO fatti oggettivi, usando esclusivamente questi 4 tipi:

        - tipo "auto": arg1 = persona, arg2 = modello dell'auto
        - tipo "colore": arg1 = oggetto, arg2 = colore
        - tipo "capitale": arg1 = paese, arg2 = città capitale
        - tipo "orario": arg1 = nome del luogo, arg2 = giorno/periodo, arg3 = ora inizio, arg4 = ora fine

        Se un'informazione non rientra in nessuno di questi tipi, ignorala.
        Usa sempre la forma più breve/canonica del nome, senza titoli.
        Se noti termini che mantengono lo stesso significato dal maschile al femminile, usa la versione al maschile (es. "nera" -> "nero").
        Utilizza sempre una notazione in minuscolo.
        Nel caso compaiano orari, usa il formato: "06:00", "09:00", "10:30", "17:20".
        Se trovi intervalli di giorni (es. "dal lunedì al venerdì"), usa "lun_ven" o "lun-ven".
        Non inventare valori non presenti nel testo.

        Frase: "{text}"
    """

    try:
        response = chat(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            format=ExtractionResult.model_json_schema(),
            options={"temperature": 0},
        )
        result = ExtractionResult.model_validate_json(response.message.content)
    except Exception as exc:
        print(f"  [Errore estrazione chunk {chunk_id}: {exc}]")
        return []

    atoms: list[str] = []
    chunk_clean = escape_asp_str(chunk_id)

    for f in result.facts:
        if f.tipo == "auto":
            a1 = escape_asp_str(norm_string(f.arg1))
            a2 = escape_asp_str(norm_string(f.arg2))
            atoms.append(f'auto("{a1}", "{a2}", "{chunk_clean}").')
        elif f.tipo == "colore":
            a1 = escape_asp_str(norm_string(f.arg1))
            a2 = escape_asp_str(norm_color(f.arg2))
            atoms.append(f'colore("{a1}", "{a2}", "{chunk_clean}").')
        elif f.tipo == "capitale":
            a1 = escape_asp_str(norm_string(f.arg1))
            a2 = escape_asp_str(norm_string(f.arg2))
            atoms.append(f'capitale("{a1}", "{a2}", "{chunk_clean}").')
        elif f.tipo == "orario" and f.arg3 and f.arg4:
            a1 = escape_asp_str(norm_string(f.arg1))
            a2 = escape_asp_str(norm_string(f.arg2))
            a3 = escape_asp_str(norm_hour(f.arg3))
            a4 = escape_asp_str(norm_hour(f.arg4))
            atoms.append(f'orario("{a1}", "{a2}", "{a3}", "{a4}", "{chunk_clean}").')

    return atoms

def build_extracted_llm(database: dict[str, str]) -> dict[str, list[str]]:
    return {
        chunk_id: extract_facts_by_chunk(chunk_id, text)
        for chunk_id, text in database.items()
    }

- build_fact(extracted): Appiattisce le liste di fatti del dizionario e le unisce in un’unica stringa, con un fatto per riga. La stringa può essere aggiunta al programma ASP.
- save_fact_file(extracted, path="fact.lp"): Salva i fatti in un file di testo con estensione `.lp`. Inserisce un commento con l’identificatore di ogni chunk prima dei relativi fatti.

In [ ]:
def build_fact(extracted: dict[str, list[str]]) -> str:
    lines = [fact for facts in extracted.values() for fact in facts]
    return "\n".join(lines)


def save_fact_file(extracted: dict[str, list[str]], path: str = "fact.lp") -> None:
    with open(path, "w", encoding="utf-8") as fp:
        for idx, facts in extracted.items():
            fp.write(f"% Chunk: {idx}\n")
            for f in facts:
                fp.write(f"{f}\n")
            fp.write("\n")

## Esecuzione del programma ASP

- make_control(program): Crea un oggetto `clingo.Control`, aggiunge il programma ASP, lo prepara (*grounding*) e restituisce il controllo pronto per la risoluzione.
- solving(ctl): Esegue il solver e raccoglie gli atomi mostrati dal modello. Se vengono prodotti più modelli, conserva quelli dell’ultimo modello visitato.
- extract_result(atoms): Organizza gli atomi restituiti da Clingo in quattro categorie: occorrenze, discordanze, elementi non coperti e riferimenti. Restituisce il risultato come oggetto `ClingoResults`.

In [ ]:
def make_control(program: str) -> clingo.Control:
    ctl = clingo.Control()
    ctl.add("base", [], program)
    ctl.ground([("base", [])])
    return ctl

def solving(ctl: clingo.Control) -> list[clingo.Symbol]:
    last_model_symbols: list[clingo.Symbol] = []

    with ctl.solve(yield_=True) as handle:
        for model in handle:
            # Sovrascrive per prendere solo il modello finale (evita duplicazioni se ci sono più modelli)
            last_model_symbols = model.symbols(shown=True)

    return last_model_symbols


def extract_result(atoms: Iterable[clingo.Symbol]) -> ClingoResults:
    occorrenze: list[clingo.Symbol] = []
    discordanze: list[clingo.Symbol] = []
    non_coperti: list[clingo.Symbol] = []
    riferimenti: list[clingo.Symbol] = []

    for atom in atoms:
        match atom.name:
            case "occorrenze":
                occorrenze.append(atom)
            case "discorde":
                discordanze.append(atom)
            case "non_coperto":
                non_coperti.append(atom)
            case "riferimento":
                riferimenti.append(atom)

    return ClingoResults(occorrenze=occorrenze, discordanze=discordanze, non_coperti=non_coperti, riferimenti=riferimenti)

## Calcolo delle metriche

- count_total_facts(extracted): Conta il numero complessivo di fatti estratti, sommando il numero di fatti associati a ciascun chunk.
- count_redundant(occorrenze): Per ogni atomo `occorrenze`, legge il conteggio finale. Quando un fatto compare più di una volta, conta come ridondanti le occorrenze oltre la prima.
- make_metrics(extracted, results): Calcola il totale dei fatti, dei riferimenti, delle ridondanze, dei conflitti e degli elementi non coperti. Da questi valori ricava le percentuali di ridondanza, conflitto e copertura, quindi restituisce un oggetto `QualityMetrics`.

In [ ]:
def count_total_facts(extracted: dict[str, list[str]]) -> int:
    return sum(len(facts) for facts in extracted.values())

def count_redundant(occorrenze: list[clingo.Symbol]) -> int:
    total = 0
    for atom in occorrenze:
        if atom.arguments and atom.arguments[-1].type == clingo.SymbolType.Number:
            val = atom.arguments[-1].number
            if val > 1:
                total += val - 1
    return total

def make_metrics(extracted: dict[str, list[str]], results: ClingoResults) -> QualityMetrics:
    total_facts = count_total_facts(extracted)
    total_references = len(results.riferimenti)
    redundant_facts = count_redundant(results.occorrenze)

    conflict = len(results.discordanze)
    uncovered = len(results.non_coperti)

    redundant_rate = (redundant_facts / total_facts * 100) if total_facts > 0 else 0.0
    conflict_rate = (conflict / total_facts * 100) if total_facts > 0 else 0.0
    coverage_rate = (
        ((total_references - uncovered) / total_references * 100) if total_references > 0 else 0.0
    )

    return QualityMetrics(
        total_facts=total_facts,
        total_references=total_references,
        redundant_facts=redundant_facts,
        conflicts=conflict,
        uncovered=uncovered,
        redundancy_rate=redundant_rate,
        conflict_rate=conflict_rate,
        coverage_rate=coverage_rate,
    )

## Visualizzazione dei risultati

- duplicate_atom(occorrenze): Restituisce solo gli atomi di occorrenza il cui conteggio è maggiore di uno
- show_conflict(discordanze): Stampa il numero di discordanze rilevate e mostra i relativi atomi
- show_duplicate(occorrenze, redundant_facts): Mostra quanti tipi di fatto risultano ripetuti e quante occorrenze superflue sono state conteggiate
- show_uncovered(non_coperti, total_references): Stampa il numero di elementi non coperti rispetto ai riferimenti attesi e li elenca
- show_quality_report(metrics, results): Compone il report finale: mostra il numero totale di fatti, conflitti, duplicati ed elementi non coperti, oltre agli indici di accuratezza, compattezza e copertura

In [ ]:
def duplicate_atom(occorrenze: list[clingo.Symbol]) -> list[clingo.Symbol]:
    return [
        atom for atom in occorrenze if atom.arguments and atom.arguments[-1].type == clingo.SymbolType.Number and atom.arguments[-1].number > 1
    ]


def show_conflict(discordanze: list[clingo.Symbol]) -> None:
    print(f"Conflitti (discordanze) rilevati: {len(discordanze)}.")
    for d in discordanze:
        print(f"  - {d}")
    print()

def show_duplicate(occorrenze: list[clingo.Symbol], redundant_facts: int) -> None:
    duplicates = duplicate_atom(occorrenze)
    print(f"Fatti ripetuti (occorrenze > 1): {len(duplicates)}, per un totale di {redundant_facts} fatti superflui.")
    for d in duplicates:
        print(f"  - {d}")
    print()

def show_uncovered(non_coperti: list[clingo.Symbol], total_references: int) -> None:
    print(f"Elementi mancanti rispetto al target ({total_references} riferimenti): {len(non_coperti)}.")
    for n in non_coperti:
        print(f"  - {n}")
    print()

def show_quality_report(metrics: QualityMetrics, results: ClingoResults) -> None:
    print("\n" + "-" * 50)
    print("Report della qualità della KB")
    print("-" * 50)
    print(f"# Fatti totali estratti: {metrics.total_facts}\n")

    show_conflict(results.discordanze)
    show_duplicate(results.occorrenze, metrics.redundant_facts)
    show_uncovered(results.non_coperti, metrics.total_references)

    print(f"Indice di accuratezza (no discordanze): {max(0.0, 100.0 - metrics.conflict_rate):.2f}%\n"
          f"Indice di compattezza (no ridondanze): {max(0.0, 100.0 - metrics.redundancy_rate):.2f}%\n"
          f"Indice di copertura (completezza): {metrics.coverage_rate:.2f}%"
    )

In [ ]:
extracted = build_extracted_llm(database)
save_fact_file(extracted)

program = f"{build_fact(extracted)}\n\n{rules}"
ctl = make_control(program)

atoms = solving(ctl)
results = extract_result(atoms)
metrics = make_metrics(extracted, results)

show_quality_report(metrics, results)